<div style="text-align: center;">

# EDA Report 

### Sales Records Dataset
</div>

### 1. Data Preparation Tasks



##### 1.1 Load the given dataset into a Pandas DataFrame

In [ ]:
import pandas as pd

df = pd.read_excel("Sales Records.xlsx")

##### 1.2 Basic Dataset Overview
- total number of rows
- total number of columns.

In [ ]:
print(f"Total number of rows: {df.shape[0]}")
print(f"Total number of columns: {df.shape[1]}")

##### 1.3 List all column names and identify their data types:
- Numerical
- Categorical
- Datetime

In [ ]:
# Conversion of Date columns to Datetime format
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# After conversion
print("\nAfter conversion of 'order date' and 'ship date' from object to 'datetime'")
print("Data Types:")
print(df.dtypes)

# Identification of column types
numerical_cols = df.select_dtypes(include=['int64', 'float64','number']).columns.tolist()
categorical_cols = df.select_dtypes(include=['string','object','category']).columns.tolist()
datetime_cols = df.select_dtypes(include=['datetime', 'datetime64']).columns.tolist()

print("\n--- Column Classification ---")
print(f"Numerical Columns ({len(numerical_cols)}):")
print(numerical_cols)

print(f"\nCategorical Columns ({len(categorical_cols)}):")
print(categorical_cols)

print(f"\nDatetime Columns ({len(datetime_cols)}):")
print(datetime_cols)

##### 1.4 Identify the target variable, if one exists in the dataset.
- Since this is an Exploratory Data Analysis (EDA) project, there is no inherent target variable; however, future business objectives could establish one—such as 'Total Profit' or 'Total Revenue' for regression tasks, or 'Sales Channel' or 'Item Type' for classification models.



##### 1.5 Explain what the dataset represents in 2-3 sentences.

- This dataset represents global sales records. Each row describes one transaction with details such as region, country, item type, sales channel, order dates, units sold, prices, costs, revenue, and profit.


##### 1.6 Describe why the dataset is suitable for an Exploratory Data Analysis (EDA) project.
- This dataset is suitable for EDA because it contains both numerical and categorical variables, along with date fields. It also has a large number of records, which makes it useful for finding patterns, distributions, correlations, and comparisons across groups.


##### 1.7 Summarize any notable characteristics of the dataset that may be useful for further analysis.
- Some notable characteristics of the dataset are that it is large, structured, and rich in business-related variables. The financial columns are closely related to each other, and the categorical columns allow comparison across regions, countries, item types, and sales channels. These features make the dataset useful for further analysis.

### 2. Data Cleaning Tasks



#### 2.1 Identify Missing Values


##### 2.1.1 Check every column in the dataset for missing or null values.

In [112]:
missing_counts=df.isnull().sum()
print(missing_counts)


Region            0
Country           0
Item Type         0
Sales Channel     0
Order Priority    0
Order Date        0
Order ID          0
Ship Date         0
Units Sold        0
Unit Price        0
Unit Cost         0
Total Revenue     0
Total Cost        0
Total Profit      0
dtype: int64


The  check confirms that the dataset is already complete and contains 0 missing values across all columns.

##### 2.1.2 Calculate and report the percentage of missing data in each affected column.


In [113]:
missing_percentages = (missing_counts / len(df)) * 100

# Creation of a DataFrame to display the missing data summary
missing_summary = pd.DataFrame({
    'Missing Values': missing_counts,
    'Percentage (%)': missing_percentages
})

# Filter to show all columns
print("--------- Missing Values Summary ---------")
display(missing_summary)

--------- Missing Values Summary ---------


,Missing Values,Percentage (%)
Region,0,0.0
Country,0,0.0
Item Type,0,0.0
Sales Channel,0,0.0
Order Priority,0,0.0
Order Date,0,0.0
Order ID,0,0.0
Ship Date,0,0.0
Units Sold,0,0.0
Unit Price,0,0.0


##### 2.1.3 Select an appropriate treatment method for each column and justify your choice:
- Drop rows if only a small number of records are affected.
- Replace missing numerical values with the mean or median, depending on the data distribution.
- Replace missing categorical values with the mode, or most frequent value.


In [114]:

cols_with_missing = missing_counts[missing_counts > 0].index.tolist()

if len(cols_with_missing) == 0:
    print("\nResult: No missing values were detected in any column of the dataset!")
else:
    print(f"\nDetected {len(cols_with_missing)} columns with missing values. Applying treatment:")
    for col in cols_with_missing:
        if col in numerical_cols:
            # Checking distribution skewness to decide between Mean or Median
            skewness = df[col].skew()
            if abs(skewness) > 1:
                # Highly skewed: we use Median
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)
                print(f"  - Filled numerical '{col}' (skewed: {skewness:.2f}) with Median: {median_val}")
            else:
                # Fairly symmetric: we use Mean
                mean_val = df[col].mean()
                df[col] = df[col].fillna(mean_val)
                print(f"  - Filled numerical '{col}' (symmetric: {skewness:.2f}) with Mean: {mean_val:.2f}")
        elif col in categorical_cols:
            # Categorical: we use Mode
            mode_val = df[col].mode()[0]
            df[col] = df[col].fillna(mode_val)
            print(f"  - Filled categorical '{col}' with Mode: '{mode_val}'")

# Double-checking that all missing values are now resolved
print(f"\nRemaining missing values in dataset: {df.isnull().sum().sum()}")



Result: No missing values were detected in any column of the dataset!

Remaining missing values in dataset: 0


- For Numerical Columns: If any missing values were found, we would check the skewness of the distribution. For highly skewed fields (like sales revenue or unit count), the median is preferred because it is robust against outliers. For normally distributed fields, the mean is used.
- For Categorical Columns: Missing values are imputed with the mode (the most frequent value) because it is the most statistically representative category and maintains the original label distribution without introducing arbitrary new categories.

##### 2.1.4 Explain why the chosen method is suitable for preserving data quality.


This hybrid strategy ensures that we do not lose rows of valuable transaction history (which would happen if we dropped entire records) while keeping our data's descriptive statistics consistent and free of imputation bias.

#### 2.2 Remove all duplicate entries.

##### 2.2.1 Identify any duplicate rows in the dataset.

In [ ]:
duplicate_rows = df.duplicated()
num_duplicates = duplicate_rows.sum()

print("--- Duplicate Record Analysis ---")

print(f"Number of duplicate rows found: {num_duplicates}")

##### 2.2.2 Remove all duplicate entries.

In [ ]:
# Identification of orgianl shape
shape_before = df.shape
print(f"Dataset shape before removing duplicates: {shape_before[0]} rows x {shape_before[1]} columns")

# We keep the 'first' occurrence and drop subsequent duplicates
df_cleaned = df.drop_duplicates(keep='first')

##### 2.2.3 Report:
- Number of duplicate rows found.
- Number of duplicate rows removed.
- New dataset shape, in rows x columns, after duplicate removal.

In [ ]:
shape_after = df_cleaned.shape
num_removed = shape_before[0] - shape_after[0]


print("--------------REPORT--------------")
print(f"Number of duplicate rows found: {num_duplicates}")
print(f"Number of duplicate rows removed: {num_removed}")
print(f"New dataset shape: {shape_after[0]} rows x {shape_after[1]} columns")

# Verification that no duplicates remain
print(f"Remaining duplicate rows: {df_cleaned.duplicated().sum()}")



#### 2.3 Detect and Handle Outliers


##### 2.3.1 Select at least two numerical columns for outlier analysis.

In [ ]:
# 'Units Sold' and 'Total Profit' are excellent choices for global sales records
outlier_cols = ['Units Sold', 'Total Profit']

##### 2.3.2 Use the Interquartile Range (IQR) method to detect outliers.



In [ ]:
class OutlierAnalyzer:
    def __init__(self, dataframe, column_name):
        self.df = dataframe
        self.column_name = column_name
        self.data = dataframe[column_name]
        
        # Calculation of Q1 and Q3 during initialization
        self.q1 = self.data.quantile(0.25)
        self.q3 = self.data.quantile(0.75)

    def iqr(self):
        return self.q3 - self.q1

    def lower_bound(self):
        iqr = self.iqr()
        return self.q1 - 1.5 * iqr

    def upper_bound(self):
        iqr = self.iqr()
        return self.q3 + 1.5 * iqr

    def count_outliers(self):
        lower = self.lower_bound()
        upper = self.upper_bound()
        
        # Count values outside the thresholds
        outliers = self.data[(self.data < lower) | (self.data > upper)]
        return len(outliers)

 

# IQR of "Unit Sold"
units_sold_analyzer = OutlierAnalyzer(df_cleaned,outlier_cols[0])
IQR_units_sold= units_sold_analyzer.iqr()
print(f"IQR of {outlier_cols[0]} is {IQR_units_sold:.2f}")

# IQR of "Total Profit"
total_profit_analyzer = OutlierAnalyzer(df_cleaned,outlier_cols[1])
IQR_total_profit= total_profit_analyzer.iqr()
print(f"IQR of {outlier_cols[1]} is {IQR_total_profit:.2f}")


##### 2.3.3 For each selected numerical column :

- Report the lower and upper bounds.


In [ ]:

# Lower and upper bound of "Units Sold"
print(f"\n----- {outlier_cols[0]} -----")
LB_Units_Sold = units_sold_analyzer.lower_bound()
print(f" Lower bound : {LB_Units_Sold}")

UB_Units_Sold = units_sold_analyzer.upper_bound()
print(f" Lower bound :  {LB_Units_Sold}")




# Lower and upper bound of "Total Profit"
print(f"\n----- {outlier_cols[1]} -----")
LB_Total_Profit = total_profit_analyzer.lower_bound()
print(f" Lower bound : {LB_Total_Profit}")

UB_Total_Profit= total_profit_analyzer.upper_bound()
print(f" Upper  bound : {UB_Total_Profit}")


- State the number of outliers detected.


In [111]:
nous = units_sold_analyzer.count_outliers()
print(f"number of outliers in {outlier_cols[0]} : {nous}")

notp = total_profit_analyzer.count_outliers()
print(f"number of outliers in {outlier_cols[1]} : {notp}")



number of outliers in Units Sold : 0
number of outliers in Total Profit : 20867


- Explain the chosen treatment:
  - Remove the outliers.
  - Cap the extreme values.
  - Retain them if they represent legitimate observations.

- Justify the decision based on the dataset context.

#### 2.4 Summary of Data Cleaning
- 

### 3. Analysis and Insights


#### 3.1 Single Column Analysis
-

#### 3.2 Relationship Between Two Columns


#### 3.3 Looking at Multiple Columns Together
-

#### 3.4 Summary of Findings
-